# Regressor training data: median time per op vs. num_tokens

Dataset: the final linear_op collection (job 21539, settle8, after all run-position fixes) — the exact file the regressors are trained on
(`2026-09-23_0949_dense_workbacklog_settle8/linear_op.csv`, md5 `3635e4d3…`, verified on load).
One figure per op. Each point is the profiler's `time_stats.<op>.median` (ms) for one (num_tokens, TP) cell; one line per TP.
Left: full range. Right: zoom on num_tokens ≤ 2048 (the dense part of the grid).

In [ ]:
import os, sys
from pathlib import Path
root = Path.cwd()
while not (root / "profiling_knowledge").exists(): root = root.parent
os.chdir(root); sys.path.insert(0, str(root))
import matplotlib.pyplot as plt
from profiling_knowledge.qwen3_30b_a3b_mi355x_profiling.regressor_bench import dataset as D

raw = D.load_raw()          # verifies md5
t = D.tidy(raw)             # long table: op, tp, num_tokens, y (ms)
print(len(raw), "rows;", t["num_tokens"].nunique(), "token values; TPs", sorted(t.tp.unique()))
t.groupby(["op","tp"]).y.agg(["count","min","median","max"]).round(4)

In [ ]:
ZOOM = 2048
for op in D.ALL_OPS:
    d = t[t.op == op]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
    for ax, zoom in zip(axes, (False, True)):
        for tp, g in d.groupby("tp"):
            g = g[g.num_tokens <= ZOOM] if zoom else g
            ax.plot(g.num_tokens, g.y, lw=1, marker="." if zoom else None, ms=2, label=f"TP{tp}")
        ax.set_xlabel("num_tokens"); ax.set_ylabel("median time (ms)")
        ax.set_title(f"{op}" + (f" (num_tokens ≤ {ZOOM})" if zoom else " (full range)"))
        ax.grid(alpha=.3); ax.legend()
    fig.tight_layout(); plt.show()